# DistilBERT Two-Model Cascade (Label1 gate + Rest multiclass) — End-to-End
Generated: 2025-10-18T00:26:50

In [ ]:

# ===== 0) Setup (installs) =====
%pip -q install "transformers>=4.44" "datasets>=2.20" "evaluate>=0.4.2" "accelerate>=0.33" scikit-learn matplotlib --upgrade


In [ ]:

# ===== 1) Imports & config =====
import os, random, math, json
from pathlib import Path
from typing import List, Dict, Any
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          EarlyStoppingCallback, pipeline)
import evaluate
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score
import matplotlib.pyplot as plt
from collections import Counter
from scipy.special import softmax

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

MODEL_CKPT = "distilbert-base-uncased"
TEXT_COL = "text"
LABEL_COL = "label"

id2label = {0:"Label4", 1:"Label5", 2:"Label2", 3:"Label3", 4:"Label1"}
label2id = {v:k for k,v in id2label.items()}
LABEL1_ID = 4

rest_ids = [i for i in range(len(id2label)) if i != LABEL1_ID]
rest_globalid2localid = {gid: li for li, gid in enumerate(rest_ids)}
rest_localid2globalid = {li: gid for gid, li in rest_globalid2localid.items()}

BATCH_SIZE = 24
GRAD_ACCUM = 1
LR_BIN = 3e-5
LR_MC  = 3e-5
EPOCHS_BIN = 2
EPOCHS_MC  = 2
PATIENCE = 2

OUTPUT_DIR = "runs_distilbert_label1_cascade"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)


In [ ]:

# ===== 2) Load data =====
CSV_TRAIN = []
CSV_VAL   = []
CSV_TEST  = []

if len(CSV_TRAIN) > 0:
    files = {"train": CSV_TRAIN}
    if len(CSV_VAL) > 0: files["validation"] = CSV_VAL
    if len(CSV_TEST) > 0: files["test"] = CSV_TEST
    ds = load_dataset("csv", data_files=files)
else:
    ds = load_dataset("yelp_review_full")
    ds = ds.rename_columns({"text": TEXT_COL, "label": LABEL_COL})

if "validation" not in ds:
    split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
    data = DatasetDict(train=split["train"], validation=split["test"])
    data["test"] = ds["test"] if "test" in ds else data["validation"].train_test_split(test_size=0.5, seed=SEED)["test"]
else:
    data = ds

def coerce_labels(example):
    val = example[LABEL_COL]
    if isinstance(val, str):
        return {LABEL_COL: label2id.get(val, val)}
    else:
        return example

data = data.map(coerce_labels)
print(data)
print("Train label counts:", Counter(data["train"][LABEL_COL]))
print("Val   label counts:", Counter(data["validation"][LABEL_COL]))


In [ ]:

# ===== 3) Tokenization =====
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
def tok(batch):
    return tokenizer(batch[TEXT_COL], truncation=True)
data_tok = data.map(tok, batched=True, remove_columns=[TEXT_COL])
collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:

# ===== 4) Build models =====
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

def build_model(num_labels, freeze_encoder=False):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CKPT, num_labels=num_labels
    )
    if freeze_encoder:
        for p in model.distilbert.parameters():
            p.requires_grad = False
    return model


In [ ]:

# ===== 5) Prepare binary dataset (Label1 vs Rest) =====
def make_binary_split(ds_tok_split, raw_split):
    # labels: 1 if Label1 else 0
    return ds_tok_split.map(lambda ex, idx: {"labels": 1 if raw_split[idx][LABEL_COL]==LABEL1_ID else 0}, with_indices=True)

bin_train = make_binary_split(data_tok["train"], data["train"])
bin_val   = make_binary_split(data_tok["validation"], data["validation"])
bin_test  = make_binary_split(data_tok["test"], data["test"])

from collections import Counter
print("Binary train label counts:", Counter(bin_train["labels"]))


In [ ]:

# ===== 6) Train Model A (binary) =====
metric_acc = evaluate.load("accuracy")

def compute_metrics_bin(eval_pred):
    logits, labels = eval_pred
    preds = (np.argmax(logits, axis=1)).astype(int)
    out = {}
    out["accuracy"] = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    out["recall_label1"] = recall_score(labels, preds, pos_label=1, zero_division=0)
    out["precision_label1"] = precision_score(labels, preds, pos_label=1, zero_division=0)
    out["f1_label1"] = f1_score(labels, preds, pos_label=1, zero_division=0)
    return out

args_bin = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "bin_label1_gate"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR_BIN,
    num_train_epochs=EPOCHS_BIN,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
    eval_steps=1000,
    save_steps=1000,
    metric_for_best_model="f1_label1",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=GRAD_ACCUM,
    report_to="none"
)

model_bin = build_model(num_labels=2)
trainer_bin = Trainer(
    model=model_bin, args=args_bin,
    train_dataset=bin_train,
    eval_dataset=bin_val,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics_bin,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
)
trainer_bin.train()
print("Binary best model:", trainer_bin.state.best_model_checkpoint)


In [ ]:

# ===== 7) Tune τ on validation =====
preds_val_bin = trainer_bin.predict(bin_val)
logits_bin_val, labels_bin_val = preds_val_bin.predictions, preds_val_bin.label_ids
probs_val_bin = softmax(logits_bin_val, axis=1)[:,1]

best = (-1, None, None)
for tau in np.linspace(0.30, 0.80, 11):
    y_pred = (probs_val_bin >= tau).astype(int)
    f1 = f1_score(labels_bin_val, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(labels_bin_val, y_pred, pos_label=1, zero_division=0)
    score = f1
    if score > best[0]:
        best = (score, tau, (f1, rec))
best_tau = best[1]
print("Best τ (binary gate) on validation:", best_tau, " (F1, Recall)=", best[2])


In [ ]:

# ===== 8) Prepare rest dataset (remove Label1, remap to 0..3) =====
def filter_rest(ds_tok_split, raw_split):
    idx = [i for i, y in enumerate(raw_split[LABEL_COL]) if y != LABEL1_ID]
    subset_tok = ds_tok_split.select(idx)
    subset_raw = raw_split.select(idx)
    ds_local = subset_tok.map(lambda ex, j: {"labels": rest_globalid2localid[subset_raw[j][LABEL_COL]]}, with_indices=True)
    return ds_local

rest_train = filter_rest(data_tok["train"], data["train"])
rest_val   = filter_rest(data_tok["validation"], data["validation"])
rest_test  = filter_rest(data_tok["test"], data["test"])

print("Rest train label counts (local ids):", Counter(rest_train["labels"]))


In [ ]:

# ===== 9) Train Model B (multiclass on Rest) =====
def compute_metrics_rest(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    out = {}
    out["accuracy"] = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    out["f1_macro"] = f1_score(labels, preds, average="macro", zero_division=0)
    out["f1_weighted"] = f1_score(labels, preds, average="weighted", zero_division=0)
    return out

args_mc = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "mc_rest"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR_MC,
    num_train_epochs=EPOCHS_MC,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
    eval_steps=1000,
    save_steps=1000,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    load_best_model_at_end=True,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    gradient_accumulation_steps=GRAD_ACCUM,
    report_to="none"
)

model_rest = build_model(num_labels=4)
trainer_rest = Trainer(
    model=model_rest, args=args_mc,
    train_dataset=rest_train,
    eval_dataset=rest_val,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics_rest,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)]
)
trainer_rest.train()
print("Rest best model:", trainer_rest.state.best_model_checkpoint)


In [ ]:

# ===== 10) Evaluate cascade on validation and test =====
def cascade_predict(ds_tok_split, raw_split, tau):
    # binary probs
    logits_bin = trainer_bin.predict(ds_tok_split.map(lambda ex: {"labels": 0})).predictions
    p1 = softmax(logits_bin, axis=1)[:,1]
    route_label1 = (p1 >= tau)
    preds_final = np.zeros(len(p1), dtype=int)

    # For ones routed to Label1
    preds_final[route_label1] = LABEL1_ID

    # For rest, run multiclass
    idx_rest = np.where(~route_label1)[0]
    if len(idx_rest) > 0:
        subset_tok = ds_tok_split.select(idx_rest)
        logits_rest = trainer_rest.predict(subset_tok).predictions
        pred_local = np.argmax(logits_rest, axis=1)
        preds_final[idx_rest] = [rest_localid2globalid[int(x)] for x in pred_local]

    return preds_final, p1, route_label1

# Validation
preds_val, p1_val, route_val = cascade_predict(data_tok["validation"], data["validation"], best_tau)
y_val = np.array(data["validation"][LABEL_COL])

print("Validation report (cascade):")
print(classification_report(y_val, preds_val, target_names=[id2label[i] for i in range(5)], digits=3))

cm = confusion_matrix(y_val, preds_val, labels=list(range(5)))
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion Matrix — Validation (Cascade)")
plt.colorbar(im)
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels([id2label[i] for i in range(5)], rotation=45, ha="right")
ax.set_yticklabels([id2label[i] for i in range(5)])
plt.tight_layout(); plt.show()

# Test
preds_test, p1_test, route_test = cascade_predict(data_tok["test"], data["test"], best_tau)
y_test = np.array(data["test"][LABEL_COL])

print("Test report (cascade):")
print(classification_report(y_test, preds_test, target_names=[id2label[i] for i in range(5)], digits=3))

cm = confusion_matrix(y_test, preds_test, labels=list(range(5)))
fig, ax = plt.subplots(figsize=(6,6))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion Matrix — Test (Cascade)")
plt.colorbar(im)
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels([id2label[i] for i in range(5)], rotation=45, ha="right")
ax.set_yticklabels([id2label[i] for i in range(5)])
plt.tight_layout(); plt.show()


In [ ]:

# ===== 11) Top-10 highest/lowest effective loss on validation =====
# If routed to Label1: neglog = -log p1
# Else: neglog = -log p(predicted_rest_class)
logits_bin_val = trainer_bin.predict(data_tok["validation"].map(lambda ex: {"labels": 0})).predictions
p_label1_val = softmax(logits_bin_val, axis=1)[:,1]
route = (p_label1_val >= best_tau)

neglog = np.zeros(len(p_label1_val), dtype=float)
neglog[route] = -np.log(np.clip(p_label1_val[route], 1e-12, 1.0))

idx_rest = np.where(~route)[0]
if len(idx_rest) > 0:
    subset_tok = data_tok["validation"].select(idx_rest)
    logits_rest = trainer_rest.predict(subset_tok).predictions
    p_rest = np.exp(logits_rest)/np.exp(logits_rest).sum(axis=1, keepdims=True)
    pred_local = np.argmax(p_rest, axis=1)
    pick = p_rest[np.arange(len(p_rest)), pred_local]
    neglog[idx_rest] = -np.log(np.clip(pick, 1e-12, 1.0))

idx_sorted_high = np.argsort(-neglog)[:10]
idx_sorted_low  = np.argsort(neglog)[:10]

val_raw = data["validation"]
def preview(indices, title):
    print(title)
    for i in indices:
        print(f"[idx={i}] eff_neglog={neglog[i]:.4f}  true={id2label[int(y_val[i])]}  pred={id2label[int(preds_val[i])]}  routed_to_Label1={route[i]}")
        if TEXT_COL in val_raw.column_names:
            print(str(val_raw[i][TEXT_COL])[:400].replace("\n"," "))
        print("-"*80)

preview(idx_sorted_high, "Top-10 Highest Effective Loss (Validation)")
preview(idx_sorted_low,  "Top-10 Lowest Effective Loss (Validation)")


In [ ]:

# ===== 12) Save both models + simple inference helper =====
SAVE_DIR_BIN = os.path.join(OUTPUT_DIR, "bin_model_saved")
SAVE_DIR_MC  = os.path.join(OUTPUT_DIR, "rest_model_saved")
os.makedirs(SAVE_DIR_BIN, exist_ok=True)
os.makedirs(SAVE_DIR_MC, exist_ok=True)
trainer_bin.model.save_pretrained(SAVE_DIR_BIN)
trainer_rest.model.save_pretrained(SAVE_DIR_MC)
tokenizer.save_pretrained(SAVE_DIR_BIN)

clf_bin = pipeline("text-classification",
                   model=SAVE_DIR_BIN, tokenizer=SAVE_DIR_BIN,
                   return_all_scores=True, device=0 if torch.cuda.is_available() else -1)
clf_rest = pipeline("text-classification",
                    model=SAVE_DIR_MC, tokenizer=SAVE_DIR_BIN,
                    return_all_scores=True, device=0 if torch.cuda.is_available() else -1)

def predict_texts_cascade(texts, tau):
    out = []
    for s in texts:
        pb = clf_bin(s)[0]
        p1 = float(pb[1]["score"])
        if p1 >= tau:
            out.append(("Label1", 4, {"p_label1": p1}))
        else:
            pr = clf_rest(s)[0]
            probs = np.array([d["score"] for d in pr], dtype=float)
            pred_local = int(np.argmax(probs))
            pred_global = rest_localid2globalid[pred_local]
            out.append((id2label[pred_global], pred_global, {"p_label1": p1}))
    return out

samples = [
    "Outstanding service and quick resolution. Totally satisfied.",
    "It crashed again after the latest update; unusable for me.",
    "Please refund; the item arrived broken and late."
]
print(predict_texts_cascade(samples, tau=best_tau))
